1. Carga de librerías y dataset

# 1. Carga del dataset de resultados internacionales

Este dataset contiene los resultados históricos de partidos internacionales de selecciones nacionales masculinas.

Cada fila representa un partido disputado entre dos selecciones e incluye información sobre la fecha, equipos participantes, competición y resultado final.

Este será el dataset principal del proyecto, ya que contiene la variable objetivo que posteriormente se utilizará para entrenar los modelos de Machine Learning.

In [ ]:
import pandas as pd
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
results = pd.read_csv("../data/raw/results.csv")


NameError: name 'pd' is not defined

# 2. Inspección inicial

Antes de realizar cualquier transformación es necesario analizar la estructura del dataset para conocer:

- Número de filas y columnas.
- Tipos de datos.
- Valores nulos.
- Coherencia general de la información.

In [4]:
results.shape

(49287, 9)

In [5]:
results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [6]:
results.info()

<class 'pandas.DataFrame'>
RangeIndex: 49287 entries, 0 to 49286
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        49287 non-null  str    
 1   home_team   49287 non-null  str    
 2   away_team   49287 non-null  str    
 3   home_score  49215 non-null  float64
 4   away_score  49215 non-null  float64
 5   tournament  49287 non-null  str    
 6   city        49287 non-null  str    
 7   country     49287 non-null  str    
 8   neutral     49287 non-null  bool   
dtypes: bool(1), float64(2), str(6)
memory usage: 5.9 MB


In [7]:
results.describe(include="all")

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
count,49287,49287,49287,49215.000000,49215.000000,49287,49287,49287,49287
unique,16434,325,318,NaN,NaN,193,2132,269,2
top,2012-02-29,Brazil,Uruguay,NaN,NaN,Friendly,Kuala Lumpur,United States,False
freq,66,614,584,NaN,NaN,18252,739,1531,36248
mean,NaN,NaN,NaN,1.756091,1.182404,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,1.770617,1.401770,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,1.000000,0.000000,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,1.000000,1.000000,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,2.000000,2.000000,NaN,NaN,NaN,NaN


# 3. Detección de valores nulos

Se analiza la presencia de valores faltantes para determinar si es necesario realizar imputaciones o eliminar registros.

In [8]:
results.isnull().sum()

date           0
home_team      0
away_team      0
home_score    72
away_score    72
tournament     0
city           0
country        0
neutral        0
dtype: int64

# 3.1 Tratamiento de valores nulos

Se detectaron 72 registros con valores faltantes en las variables
home_score y away_score.

Dado que estas variables son necesarias para construir la variable
objetivo (goal_diff) y representan menos del 1% del total de observaciones,
se decidió eliminar dichos registros.

Esta decisión evita introducir ruido mediante técnicas de imputación
y garantiza la consistencia de los datos utilizados durante el entrenamiento.

In [9]:
results[
    results["home_score"].isna() |
    results["away_score"].isna()
].shape

(72, 9)

In [10]:
results = results.dropna(
    subset=["home_score", "away_score"]
)

In [11]:
results.isnull().sum()

date          0
home_team     0
away_team     0
home_score    0
away_score    0
tournament    0
city          0
country       0
neutral       0
dtype: int64

Se eliminaron 72 registros incompletos, lo que representa
aproximadamente un 0.15% del conjunto de datos.

Debido a la baja proporción de observaciones afectadas y a la imposibilidad
de reconstruir de forma fiable el resultado del partido, se optó por la
eliminación de dichos registros.

# 4. Detección de registros duplicados

Se verifica si existen partidos duplicados que puedan introducir sesgos en el entrenamiento de los modelos.

In [12]:
results.duplicated().sum()

np.int64(0)

# 5. Conversión de la variable fecha

La variable date se transforma al formato datetime para permitir posteriores análisis temporales y filtrados cronológicos.

In [13]:
results["date"] = pd.to_datetime(results["date"])

In [14]:
results.info()

<class 'pandas.DataFrame'>
RangeIndex: 49215 entries, 0 to 49214
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        49215 non-null  datetime64[us]
 1   home_team   49215 non-null  str           
 2   away_team   49215 non-null  str           
 3   home_score  49215 non-null  float64       
 4   away_score  49215 non-null  float64       
 5   tournament  49215 non-null  str           
 6   city        49215 non-null  str           
 7   country     49215 non-null  str           
 8   neutral     49215 non-null  bool          
dtypes: bool(1), datetime64[us](1), float64(2), str(5)
memory usage: 5.4 MB


# 6. Análisis temporal del dataset

Se analiza el periodo temporal cubierto por el dataset para determinar qué años serán utilizados durante el entrenamiento.

In [15]:
results["date"].min()

Timestamp('1872-11-30 00:00:00')

In [16]:
results["date"].max()

Timestamp('2026-03-31 00:00:00')

In [17]:
# Filtrar periodo
results_2000_2025 = results[
    (results['date'] >= '2000-01-01') &
    (results['date'] <= '2025-12-31')
]

# Número de filas
print(f"Número de partidos: {results_2000_2025.shape[0]}")

Número de partidos: 24992


# 7. Análisis de selecciones participantes

Se revisan los nombres de las selecciones para detectar posibles inconsistencias que puedan afectar a la integración con otros datasets.

In [18]:
results["home_team"].nunique()

325

In [19]:
results["away_team"].nunique()

318

In [20]:
sorted(results["home_team"].unique())

['Abkhazia',
 'Afghanistan',
 'Albania',
 'Alderney',
 'Algeria',
 'American Samoa',
 'Andalusia',
 'Andorra',
 'Angola',
 'Anguilla',
 'Antigua and Barbuda',
 'Arameans Suryoye',
 'Argentina',
 'Armenia',
 'Artsakh',
 'Aruba',
 'Australia',
 'Austria',
 'Aymara',
 'Azerbaijan',
 'Bahamas',
 'Bahrain',
 'Bangladesh',
 'Barawa',
 'Barbados',
 'Basque Country',
 'Belarus',
 'Belgium',
 'Belize',
 'Benin',
 'Bermuda',
 'Bhutan',
 'Biafra',
 'Bolivia',
 'Bonaire',
 'Bosnia and Herzegovina',
 'Botswana',
 'Brazil',
 'British Virgin Islands',
 'Brittany',
 'Brunei',
 'Bulgaria',
 'Burkina Faso',
 'Burundi',
 'Cambodia',
 'Cameroon',
 'Canada',
 'Canary Islands',
 'Cape Verde',
 'Cascadia',
 'Catalonia',
 'Cayman Islands',
 'Central African Republic',
 'Central Spain',
 'Chad',
 'Chagos Islands',
 'Chameria',
 'Chechnya',
 'Chile',
 'China PR',
 'Colombia',
 'Comoros',
 'Congo',
 'Cook Islands',
 'Corsica',
 'Costa Rica',
 'County of Nice',
 'Croatia',
 'Cuba',
 'Curaçao',
 'Cyprus',
 'Czech 

# 8. Análisis de competiciones

Se analiza la variable tournament para identificar las competiciones
presentes en el dataset.

Esta información permitirá posteriormente construir una variable que
represente la importancia relativa de cada torneo, mejorando la capacidad
predictiva de los modelos.

In [21]:
results["tournament"].value_counts()

tournament
Friendly                                18252
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                            964
                                        ...  
TIFOCO Tournament                           1
Copa Confraternidad                         1
Benedikt Fontana Cup                        1
ConIFA Challenger Cup                       1
South Asian Super Cup                       1
Name: count, Length: 193, dtype: int64

In [22]:
results["tournament"].nunique()

193

# 9. Selección preliminar de variables

Se identifican las variables potencialmente relevantes para el modelado.

Variables candidatas:

- date
- home_team
- away_team
- home_score
- away_score
- tournament
- neutral

Variables potencialmente descartables:

- city
- country

La decisión definitiva se tomará tras la fase de ingeniería de características.

# 10. Creación de la variable objetivo

La variable objetivo del proyecto será la diferencia de goles entre ambos equipos.

goal_diff = home_score - away_score

Esta variable permite representar victorias, empates y derrotas mediante una única variable numérica.

In [23]:
results["goal_diff"] = (
    results["home_score"]
    - results["away_score"]
)

In [24]:
results[
    [
        "home_score",
        "away_score",
        "goal_diff"
    ]
].head()

,home_score,away_score,goal_diff
0,0.0,0.0,0.0
1,4.0,2.0,2.0
2,2.0,1.0,1.0
3,2.0,2.0,0.0
4,3.0,0.0,3.0


# 11. Selección de variables finales

Tras el análisis exploratorio se seleccionan las variables que serán utilizadas durante las fases posteriores del proyecto.

Variables conservadas:

- date
- home_team
- away_team
- home_score
- away_score
- tournament
- neutral
- goal_diff

Las variables city y country se descartan debido a que no aportan información relevante para el objetivo del proyecto y podrían introducir una complejidad innecesaria.

In [25]:
results_clean = results[
    [
        "date",
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "tournament",
        "neutral",
        "goal_diff"
    ]
]

In [26]:
results_clean.head()

,date,home_team,away_team,home_score,away_score,tournament,neutral,goal_diff
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,False,0.0
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,False,2.0
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,False,1.0
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,False,0.0
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,False,3.0


# 12. Filtrado temporal definitivo

Con el objetivo de representar el fútbol moderno y garantizar la disponibilidad de todas las variables auxiliares (Ranking FIFA y Elo), se seleccionan únicamente los partidos disputados entre los años 2000 y 2025.

In [28]:
results_clean = results_clean[
    (results_clean["date"] >= "2000-01-01") &
    (results_clean["date"] <= "2025-12-31")
]

In [ ]:
results_clean.info()

In [29]:
results_clean["date"].min()

Timestamp('2000-01-04 00:00:00')

In [30]:
results_clean["date"].max()

Timestamp('2025-12-31 00:00:00')

In [31]:
results_clean.shape

(24992, 8)

In [32]:
results_clean.to_csv(
    "results_final.csv",
    index=False
)